# Notebook that prepares the images from COCO for the input of the Encoder

In [10]:
import os
import json
import csv
import numpy as np
from pathlib import Path
from PIL import Image
import pycocotools.mask as mask_util

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────

ANNOTATIONS_FILE = "../data/coco/annotations/instances_val2017.json"
IMAGES_DIR       = "../data/coco/val2017"
OUTPUT_DIR       = "../data/masks"
TARGET_SIZE      = 256   # px (cuadrado final)

# Clases a extraer (nombre tal cual aparece en COCO)
TARGET_CLASSES = ["person", "car", "chair", "dining table"]

# Mínimo de área del bounding box para descartar objetos muy chicos
MIN_BBOX_AREA = 1000  # píxeles^2

In [15]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def letterbox(img: Image.Image, size: int) -> Image.Image:
    """Resize manteniendo aspect ratio y rellena con negro hasta size x size."""
    img.thumbnail((size, size), Image.LANCZOS)
    padded = Image.new("L", (size, size), 0)
    offset_x = (size - img.width)  // 2
    offset_y = (size - img.height) // 2
    padded.paste(img, (offset_x, offset_y))
    return padded


def decode_annotation(ann, img_height, img_width):
    """Decodifica la segmentación COCO a una máscara binaria numpy."""
    seg = ann["segmentation"]
    if isinstance(seg, list):
        # Polígono → RLE
        rles = mask_util.frPyObjects(seg, img_height, img_width)
        rle  = mask_util.merge(rles)
    elif isinstance(seg, dict):
        # RLE comprimido (counts es string) o descomprimido (counts es lista)
        if isinstance(seg.get("counts"), list):
            # RLE descomprimido → convertir con frPyObjects
            rle = mask_util.frPyObjects(seg, img_height, img_width)
        else:
            # RLE comprimido, directo
            rle = seg
    else:
        return None
    return mask_util.decode(rle).astype(np.uint8) * 255

In [16]:

print("Cargando anotaciones COCO...")
with open(ANNOTATIONS_FILE) as f:
    coco = json.load(f)

# Mapeo id → nombre de categoría
cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
target_cat_ids = {
    c["id"] for c in coco["categories"] if c["name"] in TARGET_CLASSES
}

if len(target_cat_ids) != len(TARGET_CLASSES):
    found = {cat_id_to_name[i] for i in target_cat_ids}
    missing = set(TARGET_CLASSES) - found
    print(f"⚠️  Clases no encontradas en COCO: {missing}")

# Mapeo image_id → info de imagen
img_info = {img["id"]: img for img in coco["images"]}

# Crear carpetas de salida por clase
output_path = Path(OUTPUT_DIR)
for cls in TARGET_CLASSES:
    (output_path / cls.replace(" ", "_")).mkdir(parents=True, exist_ok=True)

# Filtrar anotaciones relevantes
annotations = [
    ann for ann in coco["annotations"]
    if ann["category_id"] in target_cat_ids
]
print(f"Anotaciones encontradas para las clases objetivo: {len(annotations)}")

metadata_rows = []
saved = 0
skipped = 0

for ann in annotations:
    # Bounding box COCO: [x, y, width, height]
    x, y, w, h = [int(v) for v in ann["bbox"]]

    # Descartar objetos muy chicos
    if w * h < MIN_BBOX_AREA:
        skipped += 1
        continue

    img_meta   = img_info[ann["image_id"]]
    img_h      = img_meta["height"]
    img_w      = img_meta["width"]
    class_name = cat_id_to_name[ann["category_id"]]
    class_dir  = class_name.replace(" ", "_")

    # Decodificar máscara completa
    mask = decode_annotation(ann, img_h, img_w)
    if mask is None:
        skipped += 1
        continue

    # Recortar con bounding box (clamp por si acaso)
    x2 = min(x + w, img_w)
    y2 = min(y + h, img_h)
    crop = mask[y:y2, x:x2]

    if crop.size == 0:
        skipped += 1
        continue

    # Resize con letterbox a 256x256
    crop_img   = Image.fromarray(crop, mode="L")
    resized    = letterbox(crop_img, TARGET_SIZE)

    # Guardar como PNG
    filename   = f"{ann['image_id']}_{ann['id']}.png"
    save_path  = output_path / class_dir / filename
    resized.save(save_path)

    metadata_rows.append({
        "filename":    str(save_path),
        "class":       class_name,
        "image_id":    ann["image_id"],
        "ann_id":      ann["id"],
        "orig_img_w":  img_w,
        "orig_img_h":  img_h,
        "bbox_x":      x,
        "bbox_y":      y,
        "bbox_w":      w,
        "bbox_h":      h,
    })
    saved += 1

    if saved % 500 == 0:
        print(f"  {saved} máscaras guardadas...")

# Guardar CSV de metadatos
csv_path = output_path / "metadata.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=metadata_rows[0].keys())
    writer.writeheader()
    writer.writerows(metadata_rows)

print(f"\n✅ Listo.")
print(f"   Guardadas : {saved}")
print(f"   Descartadas (muy chicas o inválidas): {skipped}")
print(f"   Metadata  : {csv_path}")

# Resumen por clase
print("\nDistribución por clase:")
from collections import Counter
counts = Counter(r["class"] for r in metadata_rows)
for cls, n in sorted(counts.items()):
    print(f"   {cls:<20} {n}")

Cargando anotaciones COCO...
Anotaciones encontradas para las clases objetivo: 15043
  500 máscaras guardadas...
  1000 máscaras guardadas...
  1500 máscaras guardadas...
  2000 máscaras guardadas...
  2500 máscaras guardadas...
  3000 máscaras guardadas...
  3500 máscaras guardadas...
  4000 máscaras guardadas...
  4500 máscaras guardadas...
  5000 máscaras guardadas...
  5500 máscaras guardadas...
  6000 máscaras guardadas...
  6500 máscaras guardadas...
  7000 máscaras guardadas...
  7500 máscaras guardadas...
  8000 máscaras guardadas...
  8500 máscaras guardadas...
  9000 máscaras guardadas...
  9500 máscaras guardadas...
  10000 máscaras guardadas...

✅ Listo.
   Guardadas : 10245
   Descartadas (muy chicas o inválidas): 4798
   Metadata  : ../data/masks/metadata.csv

Distribución por clase:
   bicycle              227
   car                  920
   chair                1329
   person               7769


In [17]:
import json
from collections import Counter

with open("../data/coco/annotations/instances_val2017.json") as f:
    coco = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

# Contar anotaciones por clase (con el mismo filtro de área mínima)
counts = Counter()
for ann in coco["annotations"]:
    x, y, w, h = ann["bbox"]
    if w * h >= 1000:
        counts[cat_id_to_name[ann["category_id"]]] += 1

for cls, n in counts.most_common(15):
    print(f"{cls:<25} {n}")

person                    7834
chair                     1340
car                       934
dining table              624
cup                       530
bottle                    508
book                      493
bowl                      474
umbrella                  345
truck                     336
motorcycle                317
handbag                   310
banana                    304
bench                     296
sheep                     293
